In [ ]:
import cv2
import sys
import logging as log
import datetime as dt
from time import sleep
import math

font = cv2.FONT_HERSHEY_SIMPLEX

cascPath = "C:\\Users\\HP\\Documents\\DectFace\\features\\haarcascade_frontalface_default.xml"
faceCascade = cv2.CascadeClassifier(cascPath)

eyePath ="C:\\Users\\HP\\Documents\\DectFace\\features\\haarcascade_eye.xml"
eye_cascade = cv2.CascadeClassifier(eyePath)


#upperBodyPath = "C:\\Users\\HP\\Documents\\DectFace\\features\\haarcascade_upperbody.xml"
#upperCascade = cv2.CascadeClassifier(upperBodyPath)

glassPath = "C:\\Users\\HP\\Documents\\DectFace\\features\\haarcascade_eye_tree_eyeglasses.xml"
glass_cascade =cv2.CascadeClassifier(glassPath)


log.basicConfig(filename='webcam.log',level=log.INFO)

video_capture = cv2.VideoCapture(0)
anterior = 0

width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

dectHistory = []

fps = video_capture.get(cv2.CAP_PROP_FPS)
print(fps)

In [ ]:
#History Detection Class
class HistoryDetection:
    def __init__(self, xFrom, w, yFrom, h, distance):
        self.xFrom = xFrom
        self.yFrom = yFrom
        self.centerX = xFrom + (xFrom+w - xFrom)
        self.centerY = yFrom + (yFrom + h - yFrom)
        self.distance = distance
    def isSelfEnty(self, xFrom, xTo, yFrom, yTo,noise):
        xCenter = xFrom + xTo-xFrom
        yCenter = yFrom + yTo-yFrom
        
        if (xCenter >= self.centerX - noise and xCenter <= self.centerX) or (xCenter <= self.centerX+noise and xCenter >= self.centerX):
            #if (yCenter >= self.centerY - noise and yCenter <= self.centerY) or (yCenter <= self.centerY+noise and yCenter >= self.centerY):
                #return 1
            return 1
        return 0

In [ ]:
def faceBoxHasEyes (eyes,x, y, w, h ):
    for (ex,ey,ew,eh) in eyes:
            if faceHasEyes(x, y, w, h,ex,ey,ew,eh) == 1:
                return 1
    return 0

In [ ]:
#function to find intersection of eyeboxes in face
def faceHasEyes(fx, fy, fw, fh, ex, ey, ew, eh):
    if ex >= (fx) & ex <= (fx+fw):
        if ey >= (fy) & ey <= (fy+fh):
            return 1
    return 0 

In [ ]:
def calcFaceDistance(w,h):
    distancei = (2*3.14 * 180)/(w+h*360)*1000 + 3
    distancei = distancei *2.54
    return math.floor(distancei)

In [ ]:
while True:
    if not video_capture.isOpened():
        print('Unable to load camera.')
        sleep(5)
        pass

    # Capture frame-by-frame
    ret, frame = video_capture.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = faceCascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )
    
    #detect Glasses
    #glasses = glass_cascade.detectMultiScale(
    #    gray,
    #    scaleFactor=1.1,
    #    minNeighbors=5,
    #    minSize=(30, 30)
    #)
    
    #for (x, y, w, h) in glasses:
    #    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 2)
        
    deltaX =0
    
    # Draw a rectangle around the faces
    for (x, y, w, h) in faces:
        currentMinDistance =5000
        deltaX =0
        
        #Eye detection
        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]
        eyes = eye_cascade.detectMultiScale(roi_gray)
        
        #draw Boxes for Eyes
        for (ex,ey,ew,eh) in eyes:
            cv2.rectangle(roi_color,(ex,ey),(ex+ew,ey+eh),(0,255,0),2)
        
            
        if faceBoxHasEyes(eyes,x, y, w, h):
            distancei = calcFaceDistance(w,h)
            if distancei < currentMinDistance:
                currentMinDistance = distancei
                deltaX = int(width/2) - int((x+w/2))
                dXText = "DeltaX: " + str(deltaX)
                dZText = " Distance: " + str(distancei)
                cv2.putText(frame,dXText +dZText, (5,100),font,1,(0,0,255),2)
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 242, 255), 2)
            sleep(50/1000)
            
    cv2.line(frame,(int(width/2),int(height/2)),(int((width/2))-deltaX,int(height/2)),(255,0,0),5)
    
    if anterior != len(faces):
        anterior = len(faces)
        log.info("faces: "+str(len(faces))+" at "+str(dt.datetime.now()))


    # Display the resulting frame
    cv2.imshow('Video', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    # Display the resulting frame
    cv2.imshow('Video', frame)

# When everything is done, release the capture
video_capture.release()
cv2.destroyAllWindows()